# 1. Imports y carga de datos

In [10]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/reviews_with_topics.csv')
df_temas = pd.read_csv('../data/final/reviews_topics_long.csv')


RUKITO = 'Rukito Grill&Drink - Alborada'
UMBRAL_MINIMO = 15

# 2. Reconstrucción de las tablas 

In [15]:
mapa_sentimiento = {'POS': 1, 'NEU': 0, 'NEG': -1}
df_temas['sentimiento_score'] = df_temas['sentimiento'].map(mapa_sentimiento)

tabla_sentimiento = df_temas.groupby(['restaurante', 'temas_detectados'])['sentimiento_score'].mean().unstack()
tabla_conteo = df_temas.groupby(['restaurante', 'temas_detectados']).size().unstack(fill_value=0)

# Aplicamos el umbral de confiabilidad
tabla_confiable = tabla_sentimiento.copy()
tabla_confiable[tabla_conteo < UMBRAL_MINIMO] = np.nan

tabla_confiable.round(2)

temas_detectados,ambiente_instalaciones,atencion_servicio,carne,moro,porciones,precio,sabor_comida,tiempo_espera
restaurante,,,,,,,,
Casa Res | Steak House (Mall del Sol),0.46,0.55,0.44,NaN,0.10,0.30,0.54,-0.37
Don Parrilla Steak House - Urdesa,0.48,0.48,0.08,0.00,0.06,NaN,0.48,NaN
La Casa del Tomahawk,0.17,0.26,-0.29,-0.22,-0.12,-0.14,0.28,-0.61
La Parrilla Del Ñato - Urdesa,0.40,0.70,0.44,NaN,0.06,0.23,0.66,NaN
MoroGrill - C.C. Las Terrazas,0.76,0.64,0.43,0.47,0.68,0.47,0.63,NaN
Parrillada Punta Del Este,0.56,0.77,0.83,NaN,NaN,NaN,0.69,NaN
Parrillada Restaurant El Dorado Sauces 3,0.68,0.68,0.56,NaN,0.27,0.73,0.74,NaN
Rukito Grill&Drink - Alborada,0.39,0.56,0.14,0.41,0.23,0.28,0.56,-0.02


# 3 . Rukito vs. Promedio de la competencia, por tema

In [16]:
# Separamos Rukito del resto 
rukito_scores = tabla_confiable.loc[RUKITO]
competencia_scores = tabla_confiable.drop(index=RUKITO).mean() # promedio de los otros 7

comparación = pd.DataFrame({
    'rukito': rukito_scores,
    'competencia_promedio': competencia_scores
})

comparación['brecha'] = comparación['rukito'] - comparación['competencia_promedio']
comparación = comparación.sort_values('brecha')

display(comparación.round(2))

,rukito,competencia_promedio,brecha
temas_detectados,,,
carne,0.14,0.36,-0.22
ambiente_instalaciones,0.39,0.50,-0.11
precio,0.28,0.32,-0.04
atencion_servicio,0.56,0.58,-0.02
sabor_comida,0.56,0.57,-0.01
porciones,0.23,0.18,0.06
moro,0.41,0.08,0.33
tiempo_espera,-0.02,-0.49,0.47


# 4. Ranking de restaurantes por tema 

El promedio esconde información -puede que Rukito no esté "mal vs. el promedio" pero sí sea el peor del grupo en algo. 

In [18]:
for tema in tabla_confiable.columns:
    ranking = tabla_confiable[tema].dropna().sort_values(ascending=False)
    if RUKITO in ranking.index: 
        posicion = list(ranking.index).index(RUKITO) + 1
        total = len(ranking)
        print(f'{tema:25s} -> Rukito: posicion {posicion} / {total}') 

ambiente_instalaciones    -> Rukito: posicion 7 / 8
atencion_servicio         -> Rukito: posicion 5 / 8
carne                     -> Rukito: posicion 6 / 8
moro                      -> Rukito: posicion 2 / 4
porciones                 -> Rukito: posicion 3 / 7
precio                    -> Rukito: posicion 4 / 6
sabor_comida              -> Rukito: posicion 5 / 8
tiempo_espera             -> Rukito: posicion 1 / 3


# 5. Cruzar con los datos estructurados de Google. Validación cruzada

In [19]:
ratings_estructurados = df.groupby('restaurante')[['rating_comida', 'rating_servicio', 'rating_ambiente']].mean().round(2)
ratings_estructurados['es_rukito'] = ratings_estructurados.index == RUKITO
display(ratings_estructurados.sort_values('rating_servicio'))

,rating_comida,rating_servicio,rating_ambiente,es_rukito
restaurante,,,,
La Casa del Tomahawk,3.98,3.69,4.06,False
Casa Res | Steak House (Mall del Sol),3.80,3.80,4.15,False
Don Parrilla Steak House - Urdesa,4.04,4.11,4.19,False
Rukito Grill&Drink - Alborada,4.46,4.20,4.24,True
Parrillada Punta Del Este,4.57,4.40,4.19,False
Parrillada Restaurant El Dorado Sauces 3,4.56,4.47,4.43,False
La Parrilla Del Ñato - Urdesa,4.35,4.54,4.26,False
MoroGrill - C.C. Las Terrazas,4.62,4.67,4.72,False


# 6. Tiempo de espera reportado - comparación directa

In [20]:
tiempo_espera_pct = pd.crosstab(df['restaurante'], df['tiempo_espera_reportado'], normalize='index') * 100
display(tiempo_espera_pct)

tiempo_espera_reportado,De 10 a 30 min,De 30 a 60 min,Hasta 10 min,Más de 1 hora,Sin espera
restaurante,,,,,
Casa Res | Steak House (Mall del Sol),25.000000,16.666667,8.333333,0.000000,50.000000
Don Parrilla Steak House - Urdesa,0.000000,0.000000,66.666667,0.000000,33.333333
La Casa del Tomahawk,17.777778,8.888889,27.777778,2.222222,43.333333
La Parrilla Del Ñato - Urdesa,0.000000,0.000000,16.666667,0.000000,83.333333
MoroGrill - C.C. Las Terrazas,20.000000,0.000000,30.000000,0.000000,50.000000
Parrillada Punta Del Este,33.333333,0.000000,66.666667,0.000000,0.000000
Parrillada Restaurant El Dorado Sauces 3,8.333333,8.333333,16.666667,0.000000,66.666667
Rukito Grill&Drink - Alborada,45.614035,7.017544,22.807018,7.017544,17.543860


# 7. Generador de recomendaciones accionales

In [21]:
recomendaciones = []

for tema, row in comparación.iterrows(): 
    if pd.isna(row['brecha']): 
        continue
    
    if row['brecha'] < -0.15: 
        recomendaciones.append(
            f"⚠️ OPORTUNIDAD: '{tema}' tiene un sentimiento notablemente más bajo en Rukito"
            f"({row['rukito']:.2f}) que en el promedio de competencia ({row['competencia_promedio']:.2f})"
            
        )
    elif row['brecha'] > 0.15: 
        recomendaciones.append(
            f"✅ FORTALEZA: '{tema}' es un punto donde Rukito supera claramente al promedio"
            f"({row['rukito']:.2f} vs {row['competencia_promedio']:.2f})."
        )
        
    
for r in recomendaciones:
    print(r)

⚠️ OPORTUNIDAD: 'carne' tiene un sentimiento notablemente más bajo en Rukito(0.14) que en el promedio de competencia (0.36)
✅ FORTALEZA: 'moro' es un punto donde Rukito supera claramente al promedio(0.41 vs 0.08).
✅ FORTALEZA: 'tiempo_espera' es un punto donde Rukito supera claramente al promedio(-0.02 vs -0.49).


# 8. Exportar para la creación del dashboard

In [22]:
comparación.to_csv('../data/final/rukito_vs_competencia.csv')
tabla_confiable.to_csv('../data/final/sentiment_by_topic_confiable.csv')
tiempo_espera_pct.to_csv('../data/final/tiempo_espera_comparativo.csv')
ratings_estructurados.to_csv('../data/final/ratings_estructurados.csv')

print('Insights finales guardados en data/final')

Insights finales guardados en data/final
